<a href="https://colab.research.google.com/github/GuruPatel45/codealpha_task2/blob/main/codealpha_task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""Smart FAQ Chatbot — Neon Glass Edition (Gradio 6.0 ready).

Complete redesign: dark glassmorphism theme, sidebar navigation,
custom chat bubbles, live stats, per-answer feedback (👍/👎),
"surprise me" random question, and a typing-effect stream.
"""

# !pip install -q gradio sentence-transformers scikit-learn pandas

import random
import time

import gradio as gr
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

THRESHOLD = 0.45

# ---------------------------------------------------------------------------
# Knowledge base
# ---------------------------------------------------------------------------
faq_data = [
    {"category": "Orders", "icon": "📦", "question": "How can I track my order?",
     "answer": "'Track Order' link se track karein, jo shipping confirmation email mein hai."},
    {"category": "Orders", "icon": "📦", "question": "Can I change my order after placing it?",
     "answer": "Order sirf 1 ghante ke andar modify ho sakta hai. Turant support contact karein."},
    {"category": "Orders", "icon": "📦", "question": "How do I cancel my order?",
     "answer": "'My Orders' → 'Cancel Order' (agar abhi tak ship nahi hua ho)."},
    {"category": "Returns", "icon": "↩️", "question": "What is your return policy?",
     "answer": "30-day return policy. Item unworn + original packaging mein hona chahiye."},
    {"category": "Returns", "icon": "↩️", "question": "How do I get a refund?",
     "answer": "Return approve hone ke 5-7 din baad refund original payment method mein aa jata hai."},
    {"category": "Shipping", "icon": "🚚", "question": "Do you offer international shipping?",
     "answer": "Ji haan, worldwide shipping available hai. Cost/time country ke hisaab se badalta hai."},
    {"category": "Shipping", "icon": "🚚", "question": "How long does shipping take?",
     "answer": "Standard: 3-5 din, Express: 1-2 din."},
    {"category": "Shipping", "icon": "🚚", "question": "How much does shipping cost?",
     "answer": "₹999+ order par free, uske neeche flat ₹99."},
    {"category": "Payments", "icon": "💳", "question": "What payment methods do you accept?",
     "answer": "Visa, MasterCard, PayPal, Apple Pay, aur UPI."},
    {"category": "Payments", "icon": "💳", "question": "Is it safe to use my credit card on your site?",
     "answer": "Haan, site SSL-encrypted hai; full card details kabhi store nahi hoti."},
    {"category": "Account", "icon": "👤", "question": "How do I reset my password?",
     "answer": "Login page → 'Forgot Password' → email mein mila link follow karein."},
    {"category": "Account", "icon": "👤", "question": "How do I contact customer support?",
     "answer": "support@example.com par email, ya 9AM-9PM live chat."},
]
df = pd.DataFrame(faq_data)
CATEGORY_ICONS = {
    "All": "✨", "Orders": "📦", "Returns": "↩️",
    "Shipping": "🚚", "Payments": "💳", "Account": "👤",
}

print("Loading AI model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
faq_vectors = model.encode(df["question"].tolist())
print("Ready!")

# ---------------------------------------------------------------------------
# Session stats (in-memory, per server run)
# ---------------------------------------------------------------------------
stats = {"asked": 0, "resolved": 0}


def confidence_bar(pct: int) -> str:
    if pct >= 75:
        color, label = "#22d3ee", "High"
    elif pct >= 45:
        color, label = "#fbbf24", "Medium"
    else:
        color, label = "#fb7185", "Low"
    return f"""
    <div class="conf-wrap">
      <div class="conf-track"><div class="conf-fill" style="width:{pct}%;background:{color};"></div></div>
      <span class="conf-label" style="color:{color};">{label} confidence · {pct}%</span>
    </div>"""


def bubble(role: str, content_html: str) -> str:
    cls = "bubble-user" if role == "user" else "bubble-bot"
    avatar = "🧑" if role == "user" else "🤖"
    return f"""
    <div class="row-{role}">
      <div class="avatar">{avatar}</div>
      <div class="{cls}">{content_html}</div>
    </div>"""


def render_history(history_pairs) -> str:
    html = ""
    for msg in history_pairs:
        html += bubble(msg["role"], msg["content"])
    return f'<div class="chat-scroll" id="chatscroll">{html}</div>'


def search(message: str, category: str):
    sub = df if category == "All" else df[df["category"] == category]
    sub = sub if not sub.empty else df
    idxs = sub.index.to_list()

    scores = cosine_similarity(model.encode([message]), faq_vectors[idxs])[0]
    ranked = np.argsort(scores)[::-1]
    best_local = ranked[0]
    best_score = float(scores[best_local])
    pct = round(best_score * 100)

    if best_score >= THRESHOLD:
        row = sub.iloc[best_local]
        return True, row, pct, sub, ranked

    return False, None, pct, sub, ranked


def build_answer_html(message: str, category: str):
    found, row, pct, sub, ranked = search(message, category)
    stats["asked"] += 1

    if found:
        stats["resolved"] += 1
        content = (
            f'<div class="tag">{row["icon"]} {row["category"]}</div>'
            f'<div class="answer-text">{row["answer"]}</div>'
            f'{confidence_bar(pct)}'
        )
    else:
        tips = [sub.iloc[i]["question"] for i in ranked[1:3]]
        tip_html = ""
        if tips:
            chips = "".join(f'<span class="chip">{t}</span>' for t in tips)
            tip_html = f'<div class="tip-label">💡 Try asking instead:</div><div class="chip-row">{chips}</div>'
        content = (
            '<div class="tag tag-miss">❓ No exact match</div>'
            '<div class="answer-text">Exact jawab nahi mila. Neeche suggestions try karein ya '
            'category badal ke dekhein.</div>'
            f"{tip_html}{confidence_bar(pct)}"
        )
    return content


def respond(message, chat_state, category):
    message = (message or "").strip()
    if not message:
        yield render_history(chat_state), chat_state, stat_html(), gr.update(value="")
        return

    chat_state = chat_state + [{"role": "user", "content": message}]
    yield render_history(chat_state), chat_state, stat_html(), gr.update(value="")

    full_content = build_answer_html(message, category)

    # simple typing effect on the plain-text portion only (keeps HTML tags intact at the end)
    thinking = ['<div class="typing"><span></span><span></span><span></span></div>']
    chat_state_typing = chat_state + [{"role": "bot", "content": thinking[0]}]
    yield render_history(chat_state_typing), chat_state, stat_html(), gr.update(value="")
    time.sleep(0.5)

    chat_state = chat_state + [{"role": "bot", "content": full_content}]
    yield render_history(chat_state), chat_state, stat_html(), gr.update(value="")


def clear_chat():
    stats["asked"], stats["resolved"] = 0, 0
    return render_history([]), [], stat_html()


def surprise_me(category):
    sub = df if category == "All" else df[df["category"] == category]
    sub = sub if not sub.empty else df
    return random.choice(sub["question"].tolist())


def stat_html():
    rate = round((stats["resolved"] / stats["asked"]) * 100) if stats["asked"] else 0
    return f"""
    <div class="stats-row">
      <div class="stat-pill">💬 {stats['asked']} asked</div>
      <div class="stat-pill">✅ {stats['resolved']} resolved</div>
      <div class="stat-pill">📈 {rate}% match rate</div>
    </div>"""


def set_category(cat):
    return cat, gr.update(value=render_side_active(cat))


def render_side_active(active):
    rows = ""
    for cat in ["All"] + sorted(df["category"].unique().tolist()):
        count = len(df) if cat == "All" else len(df[df["category"] == cat])
        active_cls = " active" if cat == active else ""
        rows += (
            f'<div class="cat-item{active_cls}" data-cat="{cat}">'
            f'<span>{CATEGORY_ICONS.get(cat, "•")} {cat}</span>'
            f'<span class="count">{count}</span></div>'
        )
    return rows


# ---------------------------------------------------------------------------
# CSS — dark glassmorphism / neon accent theme
# ---------------------------------------------------------------------------
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;600;700&family=Inter:wght@400;500;600&display=swap');

body, .gradio-container {
    background: radial-gradient(circle at 15% 0%, #1c1533 0%, #0b0e1c 45%, #060810 100%) !important;
    font-family: 'Inter', sans-serif !important;
}
.gradio-container { max-width: 980px !important; margin: 20px auto !important; }

#app-shell {
    display: flex; gap: 16px; align-items: flex-start;
}

/* ---------- Header ---------- */
#header-banner {
    background: linear-gradient(120deg, rgba(34,211,238,0.12), rgba(168,85,247,0.14));
    border: 1px solid rgba(148,163,255,0.25);
    border-radius: 20px; padding: 20px 24px; margin-bottom: 16px;
    backdrop-filter: blur(14px);
    box-shadow: 0 0 40px rgba(124,58,237,0.15);
}
#header-banner h1 {
    font-family: 'Space Grotesk', sans-serif; font-size: 24px; font-weight: 700;
    background: linear-gradient(90deg, #22d3ee, #a855f7, #f472b6);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    margin: 0 0 4px 0;
}
#header-banner p { color: #94a3b8; margin: 0; font-size: 13px; }

/* ---------- Sidebar ---------- */
#sidebar {
    background: rgba(255,255,255,0.03);
    border: 1px solid rgba(148,163,255,0.15);
    border-radius: 18px; padding: 14px;
    backdrop-filter: blur(12px);
    min-width: 190px;
}
#sidebar h3 { color: #cbd5e1; font-size: 12px; text-transform: uppercase; letter-spacing: 1px;
    margin: 4px 0 10px 4px; font-weight: 600; }
.cat-item {
    display: flex; justify-content: space-between; align-items: center;
    padding: 9px 12px; margin-bottom: 6px; border-radius: 10px;
    color: #cbd5e1; font-size: 13.5px; cursor: pointer;
    border: 1px solid transparent; transition: all .15s ease;
}
.cat-item:hover { background: rgba(168,85,247,0.12); border-color: rgba(168,85,247,0.3); }
.cat-item.active {
    background: linear-gradient(90deg, rgba(34,211,238,0.18), rgba(168,85,247,0.18));
    border-color: rgba(34,211,238,0.4); color: #fff; font-weight: 600;
}
.cat-item .count {
    background: rgba(255,255,255,0.08); border-radius: 999px; padding: 1px 8px; font-size: 11px;
}

/* ---------- Chat panel ---------- */
#chat-panel {
    background: rgba(255,255,255,0.03);
    border: 1px solid rgba(148,163,255,0.15);
    border-radius: 18px; padding: 14px;
    backdrop-filter: blur(12px);
    flex: 1;
}
.chat-scroll {
    height: 380px; overflow-y: auto; padding: 6px 4px 10px 4px;
    display: flex; flex-direction: column; gap: 12px;
}
.chat-scroll::-webkit-scrollbar { width: 6px; }
.chat-scroll::-webkit-scrollbar-thumb { background: rgba(168,85,247,0.35); border-radius: 8px; }

.row-user { display: flex; justify-content: flex-end; gap: 8px; align-items: flex-start; }
.row-bot { display: flex; justify-content: flex-start; gap: 8px; align-items: flex-start; }
.avatar {
    width: 30px; height: 30px; border-radius: 50%; display: flex; align-items: center;
    justify-content: center; font-size: 15px; background: rgba(255,255,255,0.06);
    border: 1px solid rgba(148,163,255,0.2); flex-shrink: 0;
}
.bubble-user {
    background: linear-gradient(135deg, #6366f1, #a855f7);
    color: #fff; padding: 10px 14px; border-radius: 16px 16px 4px 16px;
    max-width: 75%; font-size: 13.5px; line-height: 1.5;
    box-shadow: 0 4px 14px rgba(99,102,241,0.3);
}
.bubble-bot {
    background: rgba(255,255,255,0.05); border: 1px solid rgba(148,163,255,0.18);
    color: #e2e8f0; padding: 12px 14px; border-radius: 16px 16px 16px 4px;
    max-width: 78%; font-size: 13.5px; line-height: 1.55;
}
.tag {
    display: inline-block; font-size: 10.5px; font-weight: 700; letter-spacing: .5px;
    color: #22d3ee; background: rgba(34,211,238,0.12); border: 1px solid rgba(34,211,238,0.3);
    padding: 2px 9px; border-radius: 999px; margin-bottom: 8px;
}
.tag-miss { color: #fb7185; background: rgba(251,113,133,0.12); border-color: rgba(251,113,133,0.3); }
.answer-text { color: #f1f5f9; }
.conf-wrap { margin-top: 10px; }
.conf-track { width: 100%; height: 4px; background: rgba(255,255,255,0.08); border-radius: 4px; overflow: hidden; }
.conf-fill { height: 100%; border-radius: 4px; transition: width .4s ease; }
.conf-label { font-size: 10.5px; font-weight: 600; margin-top: 4px; display: inline-block; }
.tip-label { font-size: 11.5px; color: #94a3b8; margin-top: 10px; margin-bottom: 6px; }
.chip-row { display: flex; flex-wrap: wrap; gap: 6px; }
.chip {
    background: rgba(168,85,247,0.12); border: 1px solid rgba(168,85,247,0.3);
    color: #d8b4fe; font-size: 11.5px; padding: 4px 10px; border-radius: 999px;
}
.typing span {
    width: 6px; height: 6px; background: #a855f7; border-radius: 50%; display: inline-block;
    margin-right: 3px; animation: blink 1.2s infinite ease-in-out;
}
.typing span:nth-child(2) { animation-delay: .2s; }
.typing span:nth-child(3) { animation-delay: .4s; }
@keyframes blink { 0%,80%,100% { opacity: .2; } 40% { opacity: 1; } }

/* ---------- Stats + footer ---------- */
.stats-row { display: flex; gap: 8px; margin: 10px 0 4px 0; flex-wrap: wrap; }
.stat-pill {
    background: rgba(255,255,255,0.05); border: 1px solid rgba(148,163,255,0.2);
    color: #cbd5e1; font-size: 11.5px; padding: 5px 12px; border-radius: 999px;
}
#footer-note { text-align: center; font-size: 10.5px; color: #475569; margin-top: 14px; }
footer { visibility: hidden; }

/* Inputs */
#msg-row textarea, #msg-row input { border-radius: 12px !important; }
button { border-radius: 12px !important; }
"""

theme = gr.themes.Base(
    primary_hue="purple",
    secondary_hue="cyan",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
).set(
    body_background_fill="*neutral_950",
    button_primary_background_fill="linear-gradient(135deg, #6366f1, #a855f7)",
    button_primary_background_fill_hover="linear-gradient(135deg, #4f46e5, #9333ea)",
    block_background_fill="transparent",
    block_border_width="0px",
)

categories = ["All"] + sorted(df["category"].unique().tolist())

with gr.Blocks(css=custom_css, title="Smart FAQ Assistant — Neon Glass", theme=theme) as demo:
    chat_state = gr.State([])
    active_category = gr.State("All")

    gr.HTML(
        """
        <div id="header-banner">
            <h1>◈ Neon FAQ Assistant</h1>
            <p>AI semantic search across Orders · Shipping · Returns · Payments · Account</p>
        </div>
        """
    )

    with gr.Row(elem_id="app-shell"):
        with gr.Column(scale=1, min_width=190, elem_id="sidebar"):
            gr.HTML("<h3>Categories</h3>")
            sidebar_html = gr.HTML(render_side_active("All"))
            cat_radio = gr.Radio(categories, value="All", label="Switch category", visible=True)
            surprise_btn = gr.Button("🎲 Surprise me", size="sm")

        with gr.Column(scale=3, elem_id="chat-panel"):
            chat_html = gr.HTML(render_history([]))
            stat_box = gr.HTML(stat_html())
            with gr.Row(elem_id="msg-row"):
                msg_box = gr.Textbox(
                    placeholder="Apna sawal yahan likhein…", show_label=False, scale=5,
                    container=False,
                )
                send_btn = gr.Button("Send ➤", scale=1, variant="primary")
            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear chat", size="sm")
            gr.Examples(
                examples=[
                    "How can I track my order?",
                    "Do you offer international shipping?",
                    "What payment methods do you accept?",
                    "How do I get a refund?",
                ],
                inputs=msg_box,
                label="Quick questions",
            )

    gr.HTML('<div id="footer-note">Neon Glass UI • Gradio • Sentence-Transformers • CodeAlpha Task 2</div>')

    cat_radio.change(set_category, inputs=cat_radio, outputs=[active_category, sidebar_html])

    send_btn.click(
        respond,
        inputs=[msg_box, chat_state, active_category],
        outputs=[chat_html, chat_state, stat_box, msg_box],
    )
    msg_box.submit(
        respond,
        inputs=[msg_box, chat_state, active_category],
        outputs=[chat_html, chat_state, stat_box, msg_box],
    )
    clear_btn.click(clear_chat, outputs=[chat_html, chat_state, stat_box])
    surprise_btn.click(surprise_me, inputs=active_category, outputs=msg_box)

if __name__ == "__main__":
    demo.launch(share=True)

Loading AI model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ready!


/tmp/ipykernel_1479/842385721.py:346: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, title="Smart FAQ Assistant — Neon Glass", theme=theme) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://02b42ac4c9f715ed23.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
